# Data Cleaning and Similarity Feautre Vectors

# Imports and constants

In [2]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import re
import unicodedata
from typing import Dict, List

import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

cwd = Path.cwd().resolve()
NOTEBOOK_DIR = (
    cwd
    if cwd.name == "1-data-cleaning-and-feature-vector"
    else cwd / "1-data-cleaning-and-feature-vector"
)
if not NOTEBOOK_DIR.exists():
    raise FileNotFoundError(f"Couldn't locate notebook directory at {NOTEBOOK_DIR}")
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
AUDIO_BASE_COLS = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "valence",
    "tempo",
]
POPULARITY_VALUE_COLS = ["lfm_playcount", "lfm_listeners"]
POPULARITY_FEATURES = [
    "log_playcount",
    "log_listeners",
    "log_plays_per_listener",
]
TAG_TEXT_COL = "tags_clean_text"
MAX_TAGS_PER_TRACK = 10

# Helper Functions for Normalization, Tag Cleaning, and Weighting

In [3]:
REMOVE_BRACKETS_RE = re.compile(r"\([^)]*\)|\[[^\]]*\]")
NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")
SPACE_RE = re.compile(r"\s+")
TAG_CHAR_FILTER_RE = re.compile(r"[^a-z0-9#+&/-]+")

TAG_STOPLIST = {
    "album",
    "albums",
    "single",
    "singles",
    "soundtrack",
    "downloads",
    "download",
    "seen live",
    "live",
    "english",
    "spanish",
    "japanese",
    "instrumental",
    "music",
    "rock music",
    "pop music",
    "the best",
    "best",
    "favorites",
    "favorite",
    "favourite",
    "under 2000 listeners",
    "airplay",
}
TAG_CANONICAL_MAP = {
    "hip hop": "hip-hop",
    "hiphop": "hip-hop",
    "hip-hop": "hip-hop",
    "r&b": "rnb",
    "rnb": "rnb",
    "r and b": "rnb",
    "indie rock": "indie-rock",
    "alt rock": "alternative rock",
    "indie pop": "indie-pop",
    "lo fi": "lo-fi",
    "lofi": "lo-fi",
    "electro pop": "electropop",
}


def strip_accents(text: str) -> str:
    return "".join(
        ch
        for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )


def normalize_text(value: str) -> str:
    if not isinstance(value, str):
        return ""
    text = value.lower().strip()
    text = REMOVE_BRACKETS_RE.sub(" ", text)
    text = strip_accents(text)
    text = NON_ALNUM_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip()


def canonicalize_tag(tag: str) -> str:
    tag = strip_accents(tag.lower())
    tag = tag.replace("&", " and ")
    tag = tag.replace("/", " ")
    tag = TAG_CHAR_FILTER_RE.sub(" ", tag)
    tag = SPACE_RE.sub(" ", tag).strip()
    if not tag:
        return ""
    tag = TAG_CANONICAL_MAP.get(tag, tag)
    if tag.endswith("s") and len(tag) > 4:
        singular = tag[:-1]
        if singular not in TAG_STOPLIST:
            tag = singular
    return tag


def clean_tag_blob(blob: str, max_tags: int = MAX_TAGS_PER_TRACK) -> List[str]:
    if not isinstance(blob, str):
        return []
    raw_tags = [token.strip() for token in blob.split(";") if token.strip()]
    cleaned: List[str] = []
    for raw in raw_tags:
        tag = canonicalize_tag(raw)
        if not tag:
            continue
        if tag in TAG_STOPLIST:
            continue
        if tag in cleaned:
            continue
        cleaned.append(tag)
        if len(cleaned) >= max_tags:
            break
    return cleaned


def winsorize_series(series: pd.Series, upper_q: float = 0.995) -> pd.Series:
    if series.dropna().empty:
        return series
    cap = series.quantile(upper_q)
    return series.clip(lower=0, upper=cap)


def apply_block_weights(
    matrix: np.ndarray, dims: Dict[str, int], weights: Dict[str, float]
) -> np.ndarray:
    weighted = np.asarray(matrix).copy()
    start = 0
    for block in ["audio", "popularity", "tags"]:
        dim = dims.get(block, 0)
        weight = weights.get(block, 1.0)
        if dim <= 0:
            continue
        stop = start + dim
        weighted[:, start:stop] *= weight
        start = stop
    return weighted

# Cleaning and Combining

In [7]:
# Load inputs, build join keys, and dedeuplicate tracks
spotify_df = pd.read_csv(DATA_DIR / "dataset_clean.csv")
lastfm_df = pd.read_csv(DATA_DIR / "dataset_with_lastfm_all.csv")
lastfm_df = lastfm_df.drop(columns=["Unnamed: 0"], errors="ignore")

for frame in (spotify_df, lastfm_df):
    frame["norm_track"] = frame["track_name"].map(normalize_text)
    frame["norm_artist"] = frame["artists"].map(normalize_text)

search_lookup = spotify_df[
    ["norm_track", "norm_artist", "search_string"]
].drop_duplicates(subset=["norm_track", "norm_artist"], keep="first")
tracks = lastfm_df.merge(
    search_lookup,
    on=["norm_track", "norm_artist"],
    how="left",
)
print(
    f"Joined table: {len(tracks):,} rows; {tracks['track_id'].nunique():,} distinct track IDs"
)

genre_signatures = pd.read_csv(DATA_DIR / "genre_avg_signatures.csv")
genre_defaults = {
    col: genre_signatures.set_index("track_genre")[col].to_dict()
    for col in AUDIO_BASE_COLS
}
for col, mapping in genre_defaults.items():
    tracks[col] = tracks[col].fillna(tracks["track_genre"].map(mapping))

completeness_cols = AUDIO_BASE_COLS + POPULARITY_VALUE_COLS + ["lfm_duration_ms"]
tracks["missing_ratio"] = tracks[completeness_cols].isna().mean(axis=1)
before = len(tracks)
tracks = (
    tracks.sort_values(by=["missing_ratio", "popularity"], ascending=[True, False])
    .drop_duplicates(subset=["norm_artist", "norm_track"], keep="first")
    .drop(columns=["missing_ratio"])
    .reset_index(drop=True)
)
print(f"Deduplicated to {len(tracks):,} rows (dropped {before - len(tracks):,})")

tracks = tracks[tracks["track_id"].notna()].copy()
tracks["track_id"] = tracks["track_id"].astype(str)
print(f"Remaining unique tracks: {tracks['track_id'].nunique():,}")

print()

# Clean Last.fm popularity signals and align durations
for col in POPULARITY_VALUE_COLS + ["lfm_duration_ms"]:
    tracks[col] = pd.to_numeric(tracks[col], errors="coerce")
for col in POPULARITY_VALUE_COLS:
    tracks.loc[tracks[col] < 0, col] = np.nan
    tracks[col] = winsorize_series(tracks[col], upper_q=0.995)

plays_per_listener = tracks["lfm_playcount"] / tracks["lfm_listeners"].replace(
    {0: np.nan}
)
tracks["log_playcount"] = np.log1p(tracks["lfm_playcount"])
tracks["log_listeners"] = np.log1p(tracks["lfm_listeners"])
tracks["log_plays_per_listener"] = np.log1p(plays_per_listener)
tracks["plays_per_listener"] = plays_per_listener

valid_duration_mask = (
    tracks["lfm_duration_ms"].notna()
    & tracks["duration_ms"].notna()
    & (tracks["duration_ms"] > 0)
)
duration_ratio = (
    tracks.loc[valid_duration_mask, "lfm_duration_ms"]
    / tracks.loc[valid_duration_mask, "duration_ms"]
)
bad_ratio_idx = duration_ratio[(duration_ratio < 0.5) | (duration_ratio > 2.0)].index
print(
    f"Dropping {len(bad_ratio_idx):,} rows with inconsistent Spotify/Last.fm duration"
)
tracks = tracks.drop(index=bad_ratio_idx).reset_index(drop=True)

tracks = tracks[tracks["duration_ms"] > 0]
tracks = tracks[tracks["tempo"] > 0]
tracks["tempo_log"] = np.log(tracks["tempo"].clip(lower=1.0))
tracks["duration_minutes"] = tracks["duration_ms"] / 60000.0
print(
    f"After sanity checks: {len(tracks):,} rows, {tracks['track_id'].nunique():,} unique track IDs"
)

print()

# Clean and canonicalize Last.fm tags
tracks["tags_clean_list"] = tracks["lfm_tags"].apply(clean_tag_blob)
tracks["tags_clean_text"] = tracks["tags_clean_list"].apply(lambda tags: " ".join(tags))
tracks["tag_count"] = tracks["tags_clean_list"].apply(len)
tag_coverage = (tracks["tag_count"] > 0).mean()
print(f"Tag coverage: {tag_coverage:.1%} of tracks retain >=1 cleaned tag")

Joined table: 73,608 rows; 73,608 distinct track IDs
Deduplicated to 70,172 rows (dropped 3,436)
Remaining unique tracks: 70,172

Dropping 14,961 rows with inconsistent Spotify/Last.fm duration
After sanity checks: 55,129 rows, 55,129 unique track IDs

Tag coverage: 92.2% of tracks retain >=1 cleaned tag


# Cleaned, Combined, and Transformed Dataset for Modeling

In [10]:
modeling_cols = list(
    dict.fromkeys(
        [
            "track_id",
            "artists",
            "track_name",
            "track_genre",
            "search_string",
            "duration_ms",
            "tempo",
            "tempo_log",
            "duration_minutes",
            TAG_TEXT_COL,
        ]
        + AUDIO_BASE_COLS
        + POPULARITY_VALUE_COLS
        + POPULARITY_FEATURES
    )
)
modeling_df = tracks[modeling_cols + ["plays_per_listener", "tag_count"]].copy()
modeling_df[TAG_TEXT_COL] = modeling_df[TAG_TEXT_COL].fillna("")
modeling_df["track_genre"] = modeling_df["track_genre"].fillna("unknown")
genre_counts = modeling_df["track_genre"].value_counts()
rare_mask = modeling_df["track_genre"].map(genre_counts) < 2
split_labels = modeling_df["track_genre"].where(~rare_mask, other="other-rare")

clean_dataset_path = ARTIFACT_DIR / "full_dataset_clean.csv"
modeling_df.to_csv(clean_dataset_path, index=False)
size_mb = clean_dataset_path.stat().st_size / (1024 * 1024)
print(
    f"Saved cleaned dataset → {clean_dataset_path.relative_to(NOTEBOOK_DIR)} ({size_mb:.1f} MB)"
)

modeling_df.head()

Saved cleaned dataset → artifacts/full_dataset_clean.csv (19.4 MB)


,track_id,artists,track_name,track_genre,search_string,duration_ms,tempo,tempo_log,duration_minutes,tags_clean_text,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,valence,lfm_playcount,lfm_listeners,log_playcount,log_listeners,log_plays_per_listener,plays_per_listener,tag_count
0,3nqQXoyQOWXiESFLlDF1hG,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),dance,"Unholy (feat. Kim Petras) - Sam Smith, Kim Petras",156943,131.121,4.876121,2.615717,pop hyperpop electropop electronic pop rap,0.01300,0.714,0.472,0.000005,0.2660,-7.375,0.0864,0.238,10995250.0,920570.0,16.212974,13.732749,2.560629,11.943959,5
1,4uUG5RXrOk84mYEfFvj3cK,David Guetta;Bebe Rexha,I'm Good (Blue),dance,"I'm Good (Blue) - David Guetta, Bebe Rexha",175238,128.040,4.852343,2.920633,house electronic dance electro house 2022,0.00383,0.561,0.965,0.000007,0.3710,-3.673,0.0343,0.304,7924612.0,758980.0,15.885484,13.539732,2.437215,10.441134,5
2,5ww2BF9slyYgNOk37BlC4u,Manuel Turizo,La Bachata,latin,La Bachata - Manuel Turizo,162637,124.980,4.828154,2.710617,reggaeton latin pop latin pop latino bachata male vocalist colombian pop colombia colombian,0.58300,0.835,0.679,0.000002,0.2180,-5.329,0.0364,0.850,4325531.0,365331.0,15.280046,12.808562,2.552568,11.840033,10
3,6Sq7ltF9Qa7SNFBsV5Cogx,Bad Bunny;Chencho Corleone,Me Porto Bonito,latin,"Me Porto Bonito - Bad Bunny, Chencho Corleone",178567,92.005,4.521843,2.976117,bad bunny reggaeton fire latin chencho corleone,0.09010,0.911,0.712,0.000027,0.0933,-5.105,0.0817,0.425,8928528.0,632558.0,16.004762,13.357529,2.715685,14.114955,5
4,5Eax0qFko2dh7Rl2lYs3bx,Bad Bunny,Efecto,latin,Efecto - Bad Bunny,213061,98.047,4.585447,3.551017,reggaeton latin,0.14100,0.801,0.475,0.000017,0.0639,-8.797,0.0516,0.234,6119925.0,420681.0,15.627061,12.949632,2.743910,14.547662,2
